# Transformers

## 1.0 Objectives


* Translation using a fine-tuned Transformers model
* Summarisation using a fine-tune Transformers model
* Using pipeline for summarisation


Run the following cells to install and import dependencies.

In [29]:
!pip install transformers
!pip install datasets
!pip install evaluate

In [30]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, AutoTokenizer
from datasets import load_dataset
import torch
import pandas as pd
import evaluate
bleu = evaluate.load("bleu")
import string
import datasets
pd.set_option('max_colwidth', 400)

## 1.1 Translation using Text-to-Text Transfer Transformer (T5)

In [31]:
model_checkpoint = "google/flan-t5-large" # You can go to huggingface.co to select any model of your choice

In [32]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint) # Get tokeniser

In [33]:
model = T5ForConditionalGeneration.from_pretrained(model_checkpoint) # Load model from Huggingface checkpoint

In [34]:
# Load data
books_eng_fr = load_dataset("opus_books", "en-fr")
books_de_en = load_dataset("opus_books", "de-en")

In [35]:
books_eng_fr

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})

In [36]:
books_eng_fr['train']['translation'][:3]

[{'en': 'The Wanderer', 'fr': 'Le grand Meaulnes'},
 {'en': 'Alain-Fournier', 'fr': 'Alain-Fournier'},
 {'en': 'First Part', 'fr': 'PREMIÈRE PARTIE'}]

Let us filter the data and keep only 100 examples.

In [37]:
eng_fra_ds = books_eng_fr['train']['translation']
de_eng_ds = books_de_en['train']['translation']

def filter_size_limit(data_ds, min_size=4, max_size=15, k='de'):
  puncts = string.punctuation + "»«"
  filtered_dict = []
  for dict_val in data_ds:
    eng = dict_val['en']
    lang_x = dict_val[k]
    eng_len = len(eng.split(' '))
    lang_x_len = len(lang_x.split(' '))
    if eng_len>min_size and eng_len<=max_size:
      if lang_x_len>min_size and lang_x_len<=max_size:
        filtered_dict.append({'eng': eng, k: lang_x.translate(str.maketrans('', '', puncts))})
  return filtered_dict


In [38]:
df_eng_fr = pd.DataFrame(filter_size_limit(eng_fra_ds, k='fr')[:100])

In [39]:
task_prefix = "translate English to French:"

df_eng_fr['encoded_input'] = list(map(lambda x: tokenizer(f"{task_prefix} {x}", max_length=64, padding='max_length')['input_ids'], df_eng_fr['eng'].values))

In [40]:
df_eng_fr.head()

,eng,fr,encoded_input
0,"He arrived at our home on a Sunday of November, 189-.",Il arriva chez nous un dimanche de novembre 189…,"[13959, 1566, 12, 2379, 10, 216, 4363, 44, 69, 234, 30, 3, 9, 1771, 13, 1671, 6, 3, 25312, 18, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,We were living in the building of the Higher Elementary Classes at Sainte-Agathe's School.,Nous habitions les bâtiments du Cours Supérieur de SainteAgathe,"[13959, 1566, 12, 2379, 10, 101, 130, 840, 16, 8, 740, 13, 8, 16808, 18867, 23081, 44, 2788, 15, 18, 188, 5497, 88, 31, 7, 1121, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,My mother taught the infants.,Ma mère faisait la petite classe,"[13959, 1566, 12, 2379, 10, 499, 2039, 4436, 8, 9806, 7, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,Thus to-day I picture our arrival.,C’est ainsi du moins que j’imagine aujourd’hui notre arrivée,"[13959, 1566, 12, 2379, 10, 5309, 12, 18, 1135, 27, 1554, 69, 6870, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,Yet we had already been ten years in that district when Meaulnes arrived.,Nous étions pourtant depuis dix ans dans ce pays lorsque Meaulnes arriva,"[13959, 1566, 12, 2379, 10, 5201, 62, 141, 641, 118, 3, 324, 203, 16, 24, 3939, 116, 1212, 9, 83, 1496, 4363, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [41]:
df_eng_fr.shape

(100, 3)

In [42]:
reference = []
predict = []
for ref, enc in df_eng_fr[['fr', 'encoded_input']].values:
  pred = model.generate(torch.tensor([enc]))
  pred_dec = tokenizer.decode(pred[0], skip_special_tokens=True)
  reference.append(ref.lower())
  predict.append(pred_dec.lower())

In [43]:
bleu.compute(predictions=predict, references=reference)

{'bleu': 0.06823636568787546,
 'precisions': [0.2939330543933054,
  0.09345794392523364,
  0.037037037037037035,
  0.0213089802130898],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0391304347826087,
 'translation_length': 956,
 'reference_length': 920}

## 1.2 Task 1: Translation
Translate from English to German using LLM with Zero-shot prefix prompting.

### 1.2.1 Exercise 1 (Dataset Preparation)
In this section, we will prepare the English to German translation dataset. It is a dataframe with columns: 'eng', 'de' and 'encoded_input'.

**Encoded input:** tokenized indices of string `Task Prefix + English text`

The examples to be included in the dataset are from `de_eng_ds`  with 4 < length_of_example <= 15.

*The exercise requires you to encode the source sentences in the English-German dataframe.*

In [44]:
df_eng_de = pd.DataFrame(filter_size_limit(de_eng_ds)[:100])
task_prefix = "translate English to German:"
df_eng_de['encoded_input'] = list(map(lambda x: tokenizer(f"{task_prefix} {x}", max_length=64, padding='max_length')['input_ids'], df_eng_de['eng'].values)) # Type your code here

Let us take a look at one example from the dataset.

In [45]:
print(df_eng_de['eng'].values[0])

There was no possibility of taking a walk that day.


In [46]:
print(tokenizer.decode(df_eng_de['encoded_input'].values[0], skip_special_tokens=True))

translate English to German: There was no possibility of taking a walk that day.


### 1.2.2 Exercise 2 (Predictions)
Make predictions on the prepared dataset storing the predicted translation and the reference text for Bleu score calculation.

*The exercise requires you to decode the encodings and store them in a list.*

In [47]:
reference_de = []
predict_de = []
for ref, enc in df_eng_de[['de', 'encoded_input']].values:
  pred = model.generate(torch.tensor([enc]))
  # Type your code here
  decoded_pred = tokenizer.decode(pred[0], skip_special_tokens=True)
  reference_de.append(ref.lower())
  predict_de.append(pred_dec.lower())

In [48]:
predict_de[0], reference_de[0]

("je n'ai pas perdu trop de temps déjà et ils doivent",
 'es war ganz unmöglich an diesem tage einen spaziergang zu machen')

In [49]:
bleu.compute(predictions=predict_de, references=reference_de)

{'bleu': 0.0,
 'precisions': [0.0, 0.0, 0.0, 0.0],
 'brevity_penalty': 1.0,
 'length_ratio': 1.134020618556701,
 'translation_length': 1100,
 'reference_length': 970}

## 1.3 Task 2: Summarisation
Generate the summary of the summary_text using language model

Let us take a look at an example.

In [50]:
eg_text = """ On their fifth wedding anniversary, writing teacher Nick Dunne returns home to find his wife Amy missing. Amy's fame as the inspiration for her parents' Amazing Amy children's books ensures widespread press coverage. The media find Nick's apathy towards the disappearance suspicious.
Flashbacks in the form of diary entries show a complex reality and a deterioriating marriage. Amazing Amy was an idealized completion of the real Amy's failures. Both Nick and Amy lost their jobs in the recession and moved from New York to Nick's hometown of North Carthage, Missouri, to support his dying mother. Nick grew distant from Amy and began an affair with Andie, one of his students, while Amy became resentful towards Nick for uprooting her."""

In [51]:
prefix_summary = "Summarize:"
text_with_prefix = f"{prefix_summary} {eg_text}"


In [52]:
text_with_prefix

"Summarize:  On their fifth wedding anniversary, writing teacher Nick Dunne returns home to find his wife Amy missing. Amy's fame as the inspiration for her parents' Amazing Amy children's books ensures widespread press coverage. The media find Nick's apathy towards the disappearance suspicious.\nFlashbacks in the form of diary entries show a complex reality and a deterioriating marriage. Amazing Amy was an idealized completion of the real Amy's failures. Both Nick and Amy lost their jobs in the recession and moved from New York to Nick's hometown of North Carthage, Missouri, to support his dying mother. Nick grew distant from Amy and began an affair with Andie, one of his students, while Amy became resentful towards Nick for uprooting her."

In [53]:
tokenized = tokenizer(text_with_prefix, return_tensors='pt')['input_ids']

In [54]:
tokenized

tensor([[12198,  1635,  1737,    10,   461,    70,  8486,  1683,  7685,     6,
           913,  3145,  7486,  6393,    29,    15,  5146,   234,    12,   253,
           112,  2512, 13349,  3586,     5, 13349,    31,     7, 10393,    38,
             8,  3773,    21,   160,  1362,    31, 12879, 13349,   502,    31,
             7,  1335,   766,     7, 14047,  2785,  2591,     5,    37,   783,
           253,  7486,    31,     7,     3,     9, 24136,  1587,     8, 10587,
           663, 21641,     5,  9497,  1549,     7,    16,     8,   607,    13,
         25933, 10066,   504,     3,     9,  1561,  2669,    11,     3,     9,
            20, 10140,    52,    23,  1014,  5281,     5, 12879, 13349,    47,
            46,  1523,  1601,  6929,    13,     8,   490, 13349,    31,     7,
          3338,     7,     5,  2867,  7486,    11, 13349,  1513,    70,  2476,
            16,     8, 19917,    11,  2301,    45,   368,  1060,    12,  7486,
            31,     7, 22295,    13,  1117,  1184,  

In [55]:
summary_tokens = model.generate(tokenized, max_new_tokens=100)

In [56]:
summary_tokens

tensor([[    0,    71,  1561,  2669,    11,     3,     9,    20, 10140,    52,
          1014,  5281,    33,  5111,    16,    48, 25933,  1764,     5,     1]])

In [57]:
print(len(tokenized[0]))
print(len(summary_tokens[0]))

173
20


In [60]:
tokenizer.decode(summary_tokens[0], skip_special_tokens=True)

'A complex reality and a deteriorating marriage are revealed in this diary entry.'

### 1.3.1 Exercise 3

For every text in the list (summary_text),
    
* encode the text using the tokenizer.
* store the encodings as `encoded_arr`



In [61]:
summary_text = [
    """Barbie premiered at the Shrine Auditorium in Los Angeles on July 9, 2023,
    and was released in the United States on July 21.
    Its simultaneous release with Universal's Oppenheimer led to the "Barbenheimer" cultural phenomenon,
    which encouraged audiences to see both films as a double feature.
    The film received critical acclaim and has grossed $1.44 billion, becoming the highest-grossing
    film of 2023 as well as the highest-grossing film by a solo female director,
    the highest-grossing film ever released by Warner Bros., and the 14th highest-grossing film of all time.""",

    """In 1942, Oppenheimer was recruited to work on the Manhattan Project, and in 1943 he was appointed director
    of the project's Los Alamos Laboratory in New Mexico, tasked with developing the first nuclear weapons.
    His leadership and scientific expertise were instrumental in the project's success. On July 16, 1945,
    he was present at the first test of the atomic bomb, Trinity. In August 1945, the weapons were used against
    Japan in the bombings of Hiroshima and Nagasaki, the only use of nuclear weapons in an armed conflict.""",

    """Linguist Louise Banks's daughter Hannah dies at the age of twelve from an incurable illness.
Twelve extraterrestrial spacecraft hover over various locations around the Earth. In the ensuing widespread panic,
affected nations send military and scientific experts to monitor and study them. In the United States,
US Army Colonel Weber recruits Banks and physicist Ian Donnelly to study the craft above Montana.
On board, Banks and Donnelly make contact with two cephalopod-like, seven-limbed aliens,
whom they call "heptapods"; Donnelly nicknames them Abbott and Costello.
Banks and Donnelly research the complex written language of the aliens,
consisting of palindromic phrases written with circular symbols,
and share the results with other nations. As Banks studies the language, she starts to have
flashback-like visions of her daughter."""
]

Add the prefix to each text.

Tokenize it using the tokenizer(...).

Get the 'input_ids' from the tokenizer output.

Store all tokenized results in a list called encoded_arr.

In [62]:
encoded_arr = [] # Type your code here
prefix_summary = "Summarize:"
for text in summary_text:
    text_with_prefix = f"{prefix_summary} {text}"
    tokenized = tokenizer(text_with_prefix, return_tensors='pt')['input_ids']
    encoded_arr.append(tokenized)

In [63]:
encoded_arr

[tensor([[12198,  1635,  1737,    10, 27945, 13539,    26,    44,     8,  4488,
           9249, 19895,    32, 11879,    16,  3144,  4975,    30,  1718,  9902,
            460,  2773,     6,    11,    47,  1883,    16,     8,   907,  1323,
             30,  1718,  1401,     5,    94,     7, 28612,  1576,    28, 12489,
             31,     7,  4495,  3208,  3254,    49,  2237,    12,     8,    96,
            279,     9, 10222,  3254,    49,   121,  2779, 15037,     6,    84,
           6470, 11209,    12,   217,   321,  4852,    38,     3,     9,  1486,
           1451,     5,    37,   814,  1204,  2404,     3,     9,    75, 15085,
             11,    65, 16917,    26,  1970,     5,  3628,  2108,     6,  2852,
              8,  2030,    18,  3844,     7,     7,    53,   814,    13,   460,
           2773,    38,   168,    38,     8,  2030,    18,  3844,     7,     7,
             53,   814,    57,     3,     9,  6729,  3955,  2090,     6,     8,
           2030,    18,  3844,     7,   

### 1.3.2 Exercise 4
Generate text (prediction) and decode the generated text.

In [65]:
summary = []
for enc in encoded_arr:
    # Type the remaining part of the codes after this line
    summary_tokens = model.generate(enc, max_new_tokens=100)
    decoded_summary = tokenizer.decode(summary_tokens[0], skip_special_tokens=True)
    summary.append(decoded_summary)

In [66]:
summary_text[1]

"In 1942, Oppenheimer was recruited to work on the Manhattan Project, and in 1943 he was appointed director\n    of the project's Los Alamos Laboratory in New Mexico, tasked with developing the first nuclear weapons.\n    His leadership and scientific expertise were instrumental in the project's success. On July 16, 1945,\n    he was present at the first test of the atomic bomb, Trinity. In August 1945, the weapons were used against\n    Japan in the bombings of Hiroshima and Nagasaki, the only use of nuclear weapons in an armed conflict."

In [67]:
summary[1]

'He was the first to work on the development of nuclear weapons.'

## 1.4 Summarization with a fine-tuned model

In [68]:
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [69]:
eg_text

" On their fifth wedding anniversary, writing teacher Nick Dunne returns home to find his wife Amy missing. Amy's fame as the inspiration for her parents' Amazing Amy children's books ensures widespread press coverage. The media find Nick's apathy towards the disappearance suspicious.\nFlashbacks in the form of diary entries show a complex reality and a deterioriating marriage. Amazing Amy was an idealized completion of the real Amy's failures. Both Nick and Amy lost their jobs in the recession and moved from New York to Nick's hometown of North Carthage, Missouri, to support his dying mother. Nick grew distant from Amy and began an affair with Andie, one of his students, while Amy became resentful towards Nick for uprooting her."

In [70]:
summary = summarizer(eg_text)

In [71]:
summary[0]["summary_text"]

"On their fifth wedding anniversary, Nick Dunne returns home to find his wife Amy missing. Amy's fame as the inspiration for her parents' Amazing Amy children's books ensures widespread press coverage. Nick grew distant from Amy and began an affair with Andie, one of his students, while Amy became resentful towards Nick for uprooting her."

## 1.5 Exercise 5
Using the pipeline, summarize the text in `summary_text`.

Compare your results with zero-shot summarization in the previous exercise. What do you think of both results?

In [72]:
# Type your code below this line
summaries = []
for text in summary_text:
    summariser = summarizer(text, max_length=24)
    summaries.append(summariser[0]["summary_text"])

Your min_length=56 must be inferior than your max_length=24.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1633: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (24). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(
Your min_length=56 must be inferior than your max_length=24.
Your min_length=56 must be inferior than your max_length=24.


In [73]:
summaries

['Barbie premiered at the Shrine Auditorium in Los Angeles on July 9, 2023, and was released',
 'In 1942, Oppenheimer was recruited to work on the Manhattan Project. In 1943 he was appointed director',
 "Linguist Louise Banks's daughter Hannah dies at the age of twelve from an incurable illness. Banks"]

# This is the end of the lesson.